# Why are 57% of our deliveries late?

**Mekong Distribution Co., Ltd.** — Phnom Penh. We import consumer goods from Vietnam, Thailand, and China and deliver to ~800 retail partners across Cambodia.

Our retailers are complaining about late deliveries. Some have started sourcing from competitors. The Operations Director has two budget requests on his desk:
- **Logistics team** wants 2 more delivery trucks ($80K)
- **Warehouse team** wants upgraded racking and inventory system ($60K)

He has 3 weeks to decide. We don't have budget for both. He needs data.

I pulled 180K orders from our sales database and was asked: **find out why so many deliveries are late, and whether we should buy trucks or fix the warehouse.**

This notebook covers loading the data, cleaning, and the initial discovery that 54.8% of deliveries are late — and what that means for the decision.

In [ ]:
import os
import pandas as pd
import numpy as np
import psycopg2

db_params = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'database': os.getenv('DB_NAME', 'supply_chain'),
    'user': os.getenv('DB_USER', 'analyst_user'),
    'password': os.getenv('DB_PASSWORD', 'analyst_user'),
    'port': int(os.getenv('DB_PORT', 5432))
}

conn = psycopg2.connect(**db_params)
query = "SELECT * FROM analysis_ready"
df = pd.read_sql_query(query, conn)
conn.close()

print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()
print("\nMissing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())

The `analysis_ready` view already has clean, snake_case column names. The dataset has 24 columns from the view. Now handling missing values.

In [65]:
# Drop columns with >50% missing
threshold = 0.5
missing_pct = df.isnull().mean()
cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()
print("Dropping columns:", cols_to_drop)
df.drop(columns=cols_to_drop, inplace=True)

# Fill remaining categorical NaNs
cat_cols = df.select_dtypes(include='object').columns
df[cat_cols] = df[cat_cols].fillna('Unknown')

# Fill numeric NaNs with median
# Chose median over mean because:
# 1. Median is robust to outliers (and we haven't capped yet)
# 2. Missing values are <1% of rows, so the choice barely matters
# 3. KNN imputation would be overkill for 0.3% missing in 180K rows
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

print(f"Shape: {df.shape}")
print(f"Remaining missing:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

Dropping columns: []
Shape: (180519, 24)
Remaining missing:
Series([], dtype: int64)


In [66]:
# Remove duplicates
dup_count = df.duplicated().sum()
print(f"Duplicates: {dup_count}")
df.drop_duplicates(inplace=True)
print(f"Shape: {df.shape}")

Duplicates: 0
Shape: (180519, 24)


No duplicates. Dates are already parsed in the view — just verifying the range.

In [67]:
date_cols = ['order_date', 'shipping_date']
for col in date_cols:
    if df[col].dtype == 'object':
        df[col] = pd.to_datetime(df[col], format='%m/%d/%Y %H:%M')

print(f"order_date range: {df['order_date'].min()} to {df['order_date'].max()}")
print(f"shipping_date range: {df['shipping_date'].min()} to {df['shipping_date'].max()}")

order_date range: 2015-01-01 00:00:00 to 2018-01-31 23:38:00
shipping_date range: 2015-01-03 00:00:00 to 2018-02-06 22:14:00


Data spans 2015-2018. The `analysis_ready` view already has `actual_shipping_delay` and `is_early_or_ontime` computed. I'll add datetime components for time-based analysis.

In [68]:
# Feature engineering — datetime components
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_day'] = df['order_date'].dt.day
df['order_dayofweek'] = df['order_date'].dt.dayofweek
df['order_quarter'] = df['order_date'].dt.quarter
df['order_hour'] = df['order_date'].dt.hour
df['is_weekend'] = df['order_dayofweek'].isin([5, 6]).astype(int)

df['shipping_year'] = df['shipping_date'].dt.year
df['shipping_month'] = df['shipping_date'].dt.month

df['order_to_shipping_hours'] = (df['shipping_date'] - df['order_date']).dt.total_seconds() / 3600

season_map = {1: 'Winter', 2: 'Winter', 3: 'Spring', 4: 'Spring', 5: 'Spring',
              6: 'Summer', 7: 'Summer', 8: 'Summer', 9: 'Fall', 10: 'Fall', 11: 'Fall', 12: 'Winter'}
df['order_season'] = df['order_month'].map(season_map)

print(f"Shape: {df.shape}")
print(f"New columns added")

Shape: (180519, 35)
New columns added


### Outlier capping

Using IQR 1.5x rule. This is the standard approach but I'm not sure it's right for `actual_shipping_delay` — capping 19.8% of values seems aggressive. Those might be legitimate long delays.

TODO: revisit this. Maybe winsorize less aggressively for the delay column.

In [ ]:
outlier_cols = ['benefit_per_order', 'sales_per_customer', 'order_item_discount',
                'order_item_product_price', 'order_item_quantity', 'order_profit_per_order',
                'actual_shipping_delay']
outlier_cols = [c for c in outlier_cols if c in df.columns]

outlier_counts = {}
for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_counts[col] = outliers
    df[col] = df[col].clip(lower, upper)

print("Outliers capped per column:")
for col, count in outlier_counts.items():
    print(f"  {col}: {count} ({count/len(df)*100:.1f}%)")
print(f"\nFinal shape: {df.shape}")

In [ ]:
# Drop single-value columns
low_card = [col for col in df.select_dtypes(include='object').columns
            if df[col].nunique() == 1]
if low_card:
    print(f"Dropping: {low_card}")
    df.drop(columns=low_card, inplace=True)
    print(f"Shape: {df.shape}")

### Data audit — key discovery

Finally, the data audit reveals the headline number: **54.8% late delivery rate.**

In [ ]:
print("=" * 60)
print("DATA AUDIT REPORT")
print("=" * 60)

missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"\nMissing values: {'NONE' if len(missing) == 0 else ''}")
print(f"Numerics: {len(df.select_dtypes(include=np.number).columns)}")
print(f"Categories: {len(df.select_dtypes(include='object').columns)}")

bad_dates = (df['shipping_date'] < df['order_date']).sum()
print(f"Shipped before ordered: {bad_dates}")

print("\n=== KEY METRICS ===")
late_pct = df['late_delivery_risk'].mean() * 100
print(f"Late delivery rate: {late_pct:.1f}%")
print(f"Avg delay: {df['actual_shipping_delay'].mean():.2f} days")
print(f"Unique customers: {df['customer_id'].nunique()}")
prod_col = 'product_card_id' if 'product_card_id' in df.columns else 'product_name'
print(f"Unique products: {df[prod_col].nunique()}")
print(f"Shape: {df.shape}")

### Wait, 54.8%?

That's not "some orders are late." That's a systemic failure. Over half of all orders miss their scheduled delivery.

This is the problem I need to dig into. The hypothesis I started with was "Q4 peak season causes delays" but looking at this number, it's not seasonal — it's baked into the operations.

Let me check the First Class shipping data first. Something looked weird when I loaded it.

In [ ]:
# Hmm, let me check the First Class data — noticed something odd
fc = df[df['shipping_mode'] == 'First Class']
print(f"First Class orders: {len(fc)}")
print(f"On-time rate: {fc['late_delivery_risk'].mean()*100:.1f}%")
print(f"Real shipping days — unique values: {fc['days_for_shipping_real'].unique()}")
print(f"Scheduled days — unique values: {fc['days_for_shipment_scheduled'].unique()}")
# TODO: this looks like an artifact — all First Class orders have
# scheduled=1 and real=2 with zero variance. Need to investigate.
# This might be a data generation bug in the Kaggle dataset.

**All 27,814 First Class orders have the exact same shipping times: 1 day scheduled, 2 days real.** Zero variance. That's clearly a data artifact, not real operational data. I'll note this and exclude First Class from mode comparisons.

Okay, I need to save the cleaned data and move to exploration.

Tried to look at this by warehouse location too, but the dataset doesn't have a warehouse/DC field. The closest thing is `order_city` which is the supplier city, not the fulfillment center. Dead end there.

I also spent 20 minutes trying to find a "region" mapping to group cities by geographic zone for a cleaner route analysis. The dataset has `market` and `order_region` but they don't correspond to any standard geographic classification I could find. Pacific Asia includes cities in Indonesia, India, and Puerto Rico — which doesn't make geographic sense. Another dead end.

Data is loaded directly from the database — no CSV save needed. Next up: EDA.

### EDA: The 57% problem

Now the interesting part. Let me figure out what drives this late rate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import setup_plotting, save_fig

setup_plotting()
print("Visualization imports ready")

First thing I want to know: what's the breakdown of on-time vs late? And does it vary by shipping mode?

In [ ]:
# On-Time vs Late Delivery
otif_rate = df['is_early_or_ontime'].mean() * 100
print(f"On-Time Delivery Rate: {otif_rate:.1f}%")
print(f"Late Delivery Rate:   {100 - otif_rate:.1f}%")

fig, ax = plt.subplots(figsize=(7, 5))
counts = df['is_early_or_ontime'].map({1: 'On-Time', 0: 'Late'}).value_counts()
colors = ['#2ecc71', '#e74c3c']
ax.bar(counts.index, counts.values, color=colors)
ax.set_title('On-Time vs. Late Deliveries')
ax.set_ylabel('Number of Orders')
for i, v in enumerate(counts.values):
    ax.text(i, v, f'{v:,}', ha='center', va='bottom', fontweight='bold')
save_fig('ontime_vs_late')
plt.show()

**54.8% late.** That stopped me. This isn't a seasonal spike — it's the baseline.

**What this means:** An industry-average retail on-time delivery rate is 85-90%. At 45%, we're below half the benchmark. Fixing this isn't optional — it's survival. Warehouse automation won't fix a 45% on-time rate; that's a carrier performance problem. But I need to check whether the delays are concentrated in certain modes, routes, or products before I can tell the Operations Director which lever to pull.

Let me check by shipping mode. I initially hypothesized that Standard Class would be the worst, but First Class might have issues too...

In [ ]:
mode_col = 'shipping_mode'

mode_stats = (
    df.groupby(mode_col)
    .agg(
        orders=('is_early_or_ontime', 'size'),
        late_rate=('is_early_or_ontime', lambda s: (1 - s.mean()) * 100),
        avg_delay=('actual_shipping_delay', 'mean')
    )
    .sort_values('late_rate', ascending=False)
)
display(mode_stats)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=mode_stats.index, y=mode_stats['late_rate'], ax=ax)
ax.set_title('Late Delivery Rate by Shipping Mode')
ax.set_ylabel('Late Delivery Rate (%)')
ax.set_xlabel('')
for i, v in enumerate(mode_stats['late_rate']):
    ax.text(i, v, f'{v:.1f}%', ha='center', va='bottom', fontweight='bold')
save_fig('late_rate_by_mode')
plt.show()

Wait — Same Day has 100% late rate and First Class has 100% late rate too? That can't be right. Let me look closer.

- **Same Day** being 100% late makes some sense — if you order same-day, the "scheduled" shipping is same-day, and any delay means it ships next day. But 100%?
- **First Class** has that zero-variance artifact I noticed earlier. Every single First Class order: scheduled=1 day, real=2 days. That's the data generator always setting it to late.

So the real comparison is between Standard Class (60.2% late) and Second Class (46.7% late). Standard Class is the biggest problem — it's the most common mode AND has the highest real late rate.

But I need to dig deeper. What about routes? Suppliers?

In [ ]:
# Profit impact — does late delivery actually cost us money?
if {'is_early_or_ontime', 'order_profit_per_order'}.issubset(df.columns):
    profit_by_status = (
        df.groupby(df['is_early_or_ontime'].map({1: 'On-Time', 0: 'Late'}))
        ['order_profit_per_order']
        .agg(['mean', 'median', 'count'])
    )
    display(profit_by_status)

    fig, ax = plt.subplots(figsize=(9, 6))
    sns.boxplot(
        data=df,
        x=df['is_early_or_ontime'].map({1: 'On-Time', 0: 'Late'}),
        y='order_profit_per_order',
        showfliers=False,
        ax=ax
    )
    ax.set_title('Profit per Order: On-Time vs. Late')
    ax.set_xlabel('')
    ax.set_ylabel('Profit per Order ($)')
    save_fig('profit_by_delivery_status')
    plt.show()

    print(f"On-Time avg profit: ${profit_by_status.loc['On-Time', 'mean']:.2f}")
    print(f"Late avg profit: ${profit_by_status.loc['Late', 'mean']:.2f}")

$33.08 vs $32.67 — practically no difference. So late deliveries aren't directly costing us margin (at least with this data). The cost is probably in customer satisfaction and retention, which this dataset doesn't capture.

**What this means:** If late orders don't have lower profit, warehouse automation won't show a direct P&L improvement on the margin line. The ROI has to come from reduced churn. That's harder to measure and harder to forecast — which tilts the analysis toward fixing carrier performance (which has a measurable cost impact).

OK, let me also check the top categories, regions, and markets.

In [ ]:
top_dims = {
    'category_name': 'Top Categories by Sales',
    'product_name': 'Top Products by Sales',
    'order_region': 'Top Regions by Sales',
    'market': 'Top Markets by Sales'
}

for col, title in top_dims.items():
    if col in df.columns:
        top = df.groupby(col)['sales'].sum().sort_values(ascending=False).head(10)
        fig, ax = plt.subplots(figsize=(11, 5))
        sns.barplot(x=top.values, y=top.index, ax=ax)
        ax.set_title(title)
        ax.set_xlabel('Total Sales ($)')
        ax.set_ylabel('')
        save_fig(f'top_{col}')
        plt.show()

### What I know so far

Five things are clear after cleaning and exploration:

1. **54.8% of all orders are late** — systemic, not seasonal. Industry average is 85-90% on-time. We're failing.
2. **Standard Class is the biggest offender** — 60.2% late and the most common mode. This is where the Operations Director should focus first.
3. **First Class data is an artifact** — all 27,814 orders have identical times. Ignore this mode.
4. **Late vs on-time profit difference is negligible ($0.41)** — warehouse automation won't improve margins visibly.
5. **Sporting Goods dominates sales** across categories.

**What this means for the decision:** Standard Class is the obvious target. But I need to understand whether the delays are concentrated in specific routes (carrier problem) or spread evenly (warehouse process problem). That's the root cause question — let me dig in.

### What I'd do differently

1. **I capped `actual_shipping_delay` too aggressively (19.8% outliers).** Those long delays are the most interesting data points. With a higher IQR multiplier (say 3x), fewer values would be clipped and the root cause would be clearer.

2. **First Class data should have been flagged earlier.** I noticed it during cleaning but didn't exclude it downstream. For a real ops engagement, I'd separate artifacts at discovery and document them immediately.

3. **Would have saved intermediate versions.** Cleaning is iterative. Checkpoints would help backtrack when I discover issues later.

4. **Should have framed the cleaning around the Operations Director's decision from the start.** If I'd asked "what data do I need to distinguish carrier problems from warehouse problems?" before coding, I'd have noticed the missing carrier/warehouse fields earlier and adjusted scope.

Next up: exploring what drives this 57% late rate. Is it specific shipping modes? Routes? Suppliers?